# BEAM Benchmark — UMD JupyterHub (GPU Cluster)

Runs the BEAM benchmark using **Gemma 4 31B from the UMD GPU cluster** (gpu02 vLLM) for LLM calls and **Fireworks.ai** for embeddings only.

## Architecture
- **LLM (chat/extraction/QA):** `google/gemma-4-31B-it` via vLLM on gpu02 — **no rate limits**
- **Embeddings:** `qwen3-embedding-8b` via Fireworks API — small requests, minimal rate-limit risk
- **Judge (Phase 2):** Local Gemma by default (free, no rate limits)
- **Execution:** UMD JupyterHub (gpuyter.mind.cs.umd.edu) — persists when laptop is down
- **Storage:** GlusterFS home dir (`~/beam_results/`) — survives session disconnects

## Prerequisites
1. **Push your code changes to GitHub** before running this notebook (the notebook clones from GitHub)
2. Log in to [gpuyter.mind.cs.umd.edu](https://gpuyter.mind.cs.umd.edu) and open this notebook
3. **Switch kernel to Python 3.11 (beam)** — Kernel → Change kernel → Python 3.11 (beam)
4. Run cells top-to-bottom

## Network setup (do once in a JupyterHub terminal)
The JupyterHub container cannot reach gpu02:8000 directly. You need an SSH tunnel:

```bash
# In a JupyterHub terminal (File → New → Terminal):
ssh -L 8000:localhost:8000 nmokaria@gpu02.mind.cs.umd.edu -N &
```

Also start vLLM on gpu02 (if not already running):

```bash
ssh nmokaria@gpu02.mind.cs.umd.edu
nohup env TRITON_CACHE_DIR=/scratch/triton_cache_gemma VLLM_USE_FLASHINFER_SAMPLER=0 CUDA_VISIBLE_DEVICES=0,1 \
  python -m vllm.entrypoints.openai.api_server \
  --model google/gemma-4-31B-it \
  --tensor-parallel-size 2 --port 8000 --host 0.0.0.0 \
  --max-model-len 32768 --max-num-batched-tokens 4096 \
  --gpu-memory-utilization 0.85 \
  > /scratch/vllm_gemma.log 2>&1 &
```

## Notes
- vLLM is accessed via SSH tunnel at `localhost:8000` (not directly at gpu02)
- If JupyterHub disconnects, re-run with `--resume` to continue from checkpoints
- The SSH tunnel must be re-established if the JupyterHub session restarts

## 1. Clone repo & install dependencies

In [ ]:
import os, sys

REPO_DIR = os.path.expanduser('~/Agentic-Graph-Memory')

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/nmokaria27/Agentic-Graph-Memory.git {REPO_DIR}

%cd {REPO_DIR}

# Checkout the feature branch
!git checkout feat/vector-index-and-qa-improvements
!git pull origin feat/vector-index-and-qa-improvements

# Install dependencies using the beam env's Python (not system Python 3.7)
BEAM_PY = '/opt/conda/envs/beam/bin/python'
!{BEAM_PY} -m pip install --no-build-isolation -e . -q
!{BEAM_PY} -m pip install datasets json_repair sentence_transformers scipy -q

print(f'\n=== Dependencies installed ===')
print(f'Python: {sys.version}')
print(f'Executable: {sys.executable}')

## 2. Verify vLLM is running (via SSH tunnel)

Make sure you've done these in a **JupyterHub terminal** (File → New → Terminal):

**Terminal 1 — SSH tunnel:**
```bash
ssh -L 8000:localhost:8000 nmokaria@gpu02.mind.cs.umd.edu -N &
```

**Terminal 2 — Start vLLM on gpu02 (if not already running):**
```bash
ssh nmokaria@gpu02.mind.cs.umd.edu
nohup env TRITON_CACHE_DIR=/scratch/triton_cache_gemma VLLM_USE_FLASHINFER_SAMPLER=0 CUDA_VISIBLE_DEVICES=0,1 \
  python -m vllm.entrypoints.openai.api_server \
  --model google/gemma-4-31B-it \
  --tensor-parallel-size 2 --port 8000 --host 0.0.0.0 \
  --max-model-len 32768 --max-num-batched-tokens 4096 \
  --gpu-memory-utilization 0.85 \
  > /scratch/vllm_gemma.log 2>&1 &
```

Wait ~2-3 minutes, then run the cell below to confirm vLLM is reachable via the tunnel.

In [ ]:
import requests, time

# vLLM is accessed via SSH tunnel at 127.0.0.1:8000 (use 127.0.0.1, NOT localhost —
# localhost resolves to ::1 (IPv6) which the SSH tunnel doesn't bind to)
VLLM_HOST = '127.0.0.1'
VLLM_PORT = 8000
VLLM_URL = f'http://{VLLM_HOST}:{VLLM_PORT}'

# Poll until vLLM is ready (check every 10s, up to 5 minutes)
print(f'Checking vLLM at {VLLM_URL} (via SSH tunnel to gpu02) ...')
for i in range(30):
    try:
        resp = requests.get(f'{VLLM_URL}/v1/models', timeout=5)
        if resp.status_code == 200:
            models = resp.json().get('data', [])
            print(f'✅ vLLM ready — {len(models)} model(s) loaded:')
            for m in models:
                print(f'   {m["id"]}')
            break
    except Exception:
        pass
    print(f'  [{(i+1)*10}s] not ready yet — waiting...', end='\r')
    time.sleep(10)
else:
    print('\n❌ vLLM not reachable after 5 minutes.')
    print('   → Open a JupyterHub terminal and run:')
    print('       ssh -L 127.0.0.1:8000:127.0.0.1:8000 nmokaria@gpu02.mind.cs.umd.edu -N')
    print('   → Then re-run this cell')

## 3. Configure `.env`

Sets up the environment to use:
- **Gemma 4 31B** from gpu02 vLLM for all LLM calls (no rate limits)
- **Fireworks** for embeddings only (qwen3-embedding-8b)
- **Local Gemma** as the judge model for Phase 2 scoring

You'll need your Fireworks API key. It will be read from `~/.fireworks_api_key` if available, or you'll be prompted to enter it.

In [ ]:
import os, getpass, stat

# ── Get Fireworks API key ──────────────────────────────────────────
key_file = os.path.expanduser('~/.fireworks_api_key')
FIREWORKS_API_KEY = None

if os.path.exists(key_file):
    with open(key_file) as f:
        FIREWORKS_API_KEY = f.read().strip()
    print(f'✅ Fireworks API key loaded from {key_file}')

if not FIREWORKS_API_KEY:
    FIREWORKS_API_KEY = getpass.getpass('Enter Fireworks API key: ')
    # Save for future sessions (restrict permissions)
    with open(key_file, 'w') as f:
        f.write(FIREWORKS_API_KEY)
    os.chmod(key_file, stat.S_IRUSR | stat.S_IWUSR)
    print(f'✅ Key saved to {key_file} (chmod 600)')

# ── Write .env ─────────────────────────────────────────────────────
env = f"""# ── Hybrid: Gemma (gpu02 vLLM) + Fireworks embeddings ────────────
LLM_BACKEND=vllm
VLLM_BASE_URL=http://{VLLM_HOST}:{VLLM_PORT}/v1
VLLM_API_KEY=EMPTY
LLM_DEFAULT_MODEL=google/gemma-4-31B-it

# Embeddings via Fireworks (small requests, minimal rate-limit risk)
EMBEDDING_BASE_URL=https://api.fireworks.ai/inference/v1
EMBEDDING_API_KEY={FIREWORKS_API_KEY}
EMBEDDING_MODEL=accounts/fireworks/models/qwen3-embedding-8b

# ── Concurrency (local GPU — no rate limits, can be aggressive) ────
ENTITY_EXTRACT_WORKERS=6
RELATION_EXTRACT_WORKERS=6
LLM_RETRY_BACKOFF=2

# ── Judge model for Phase 2 scoring ───────────────────────────────
# Using local Gemma (free, no rate limits). Change to a Fireworks model
# for higher-quality judging if desired:
# BEAM_JUDGE_MODEL=accounts/fireworks/models/deepseek-v4-flash
BEAM_JUDGE_MODEL=google/gemma-4-31B-it
"""

with open('.env', 'w') as f:
    f.write(env)

print('=== .env written ===')
print(f'   LLM:          google/gemma-4-31B-it via {VLLM_HOST}:{VLLM_PORT}')
print(f'   Embeddings:   qwen3-embedding-8b via Fireworks')
print(f'   Judge:        google/gemma-4-31B-it (local)')

## 4. Verify connectivity

Check that both vLLM (gpu02) and Fireworks API are reachable from JupyterHub.

In [ ]:
import requests

# Check vLLM on gpu02
print('── vLLM (gpu02) ──')
try:
    resp = requests.get(f'{VLLM_URL}/v1/models', timeout=10)
    if resp.status_code == 200:
        models = resp.json().get('data', [])
        print(f'✅ vLLM reachable — {len(models)} model(s)')
        for m in models:
            print(f'   {m["id"]}')
    else:
        print(f'❌ vLLM error: {resp.status_code}')
except Exception as e:
    print(f'❌ vLLM unreachable: {e}')

# Quick chat test
print('\n── Chat test (Gemma) ──')
try:
    resp = requests.post(
        f'{VLLM_URL}/v1/chat/completions',
        json={
            'model': 'google/gemma-4-31B-it',
            'messages': [{'role': 'user', 'content': 'Say hello in one word.'}],
            'max_tokens': 50,
        },
        timeout=30,
    )
    if resp.status_code == 200:
        content = resp.json()['choices'][0]['message']['content']
        print(f'✅ Chat works — response: {content[:80]}')
    else:
        print(f'❌ Chat error: {resp.status_code} — {resp.text[:200]}')
except Exception as e:
    print(f'❌ Chat failed: {e}')

# Check Fireworks API
print('\n── Fireworks API (embeddings) ──')
try:
    resp = requests.get(
        'https://api.fireworks.ai/inference/v1/models',
        headers={'Authorization': f'Bearer {FIREWORKS_API_KEY}'},
        timeout=10,
    )
    if resp.status_code == 200:
        print('✅ Fireworks API reachable')
    else:
        print(f'❌ Fireworks error: {resp.status_code}')
except Exception as e:
    print(f'❌ Fireworks unreachable: {e}')

## 5. Configure run parameters

Edit these before running the benchmark.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  EDIT THESE PARAMETERS                                        ║
# ╚══════════════════════════════════════════════════════════════╝

TIER = '100K'              # Options: 100K, 500K, 1M
MAX_SAMPLES = 4            # Number of conversations (-1 = all)
MAX_QUESTIONS = -1         # Questions per type per chat (-1 = all)
CHUNK_SIZE = 3000          # Chunk size in chars (3000 recommended for 100K)
RETRIEVAL = 'hybrid'       # Options: hybrid, graph_completion, dense, lexical, chunk
MAX_CONTEXT_CHARS = ''     # Truncate conversation (empty = no truncation)

# ── Storage paths (GlusterFS home dir — persistent across sessions) ──
RESULTS_BASE = os.path.expanduser('~/beam_results')
KG_DIR = f'{RESULTS_BASE}/kg_cache/{TIER}'
CHECKPOINT_DIR = f'{RESULTS_BASE}/checkpoints/{TIER}'
RESPONSES_PATH = f'{RESULTS_BASE}/responses/beam_{TIER}_responses.json'
SCORES_PATH = f'{RESULTS_BASE}/scores/beam_{TIER}_scores.json'

# Create directories
!mkdir -p {KG_DIR} {CHECKPOINT_DIR} {RESULTS_BASE}/responses {RESULTS_BASE}/scores

# Use the beam env's Python for all benchmark commands
BEAM_PY = '/opt/conda/envs/beam/bin/python'

# Build the command
cmd = f"""
{BEAM_PY} evaluation/BEAM/run_eval.py \\
    --tier {TIER} \\
    --max-samples {MAX_SAMPLES} \\
    --max-questions {MAX_QUESTIONS} \\
    --chunk-size {CHUNK_SIZE} \\
    --retrieval {RETRIEVAL} \\
    --model google/gemma-4-31B-it \\
    --embedding-model accounts/fireworks/models/qwen3-embedding-8b \\
    --save-kg-dir {KG_DIR} \\
    --checkpoint-dir {CHECKPOINT_DIR} \\
    --resume \\
    --output {RESPONSES_PATH}
"""
if MAX_CONTEXT_CHARS:
    cmd += f"    --max-context-chars {MAX_CONTEXT_CHARS} \\\n"

print('Run command:')
print(cmd)
print(f'\nKG cache     → {KG_DIR}')
print(f'Checkpoints  → {CHECKPOINT_DIR}')
print(f'Responses    → {RESPONSES_PATH}')
print(f'Scores       → {SCORES_PATH}')

## 6. Run BEAM Phase 1 — Build KG + Answer Questions

**This is the long-running cell.** Keep the JupyterHub tab open.

With the local GPU cluster (no rate limits) and the batch size fix, this should take **~30-60 min per conversation** instead of 12+ hours.

If JupyterHub disconnects:
- Results saved so far are in `~/beam_results/` (GlusterFS, persistent)
- Checkpoints are in `~/beam_results/checkpoints/`
- Re-run this cell — `--resume` will skip completed stages

In [ ]:
import time

start = time.time()
print(f'Starting BEAM Phase 1 at {time.strftime("%H:%M:%S")}')
print(f'Tier: {TIER} | Samples: {MAX_SAMPLES} | Retrieval: {RETRIEVAL}')
print(f'Model: google/gemma-4-31B-it (gpu02 vLLM via SSH tunnel)')
print(f'Embeddings: qwen3-embedding-8b (Fireworks)')
print('=' * 60)

!{cmd.strip()}

elapsed = time.time() - start
hours = elapsed / 3600
print(f'\n{"=" * 60}')
print(f'Phase 1 complete in {hours:.1f} hours')
print(f'Responses saved to: {RESPONSES_PATH}')

## 7. Run BEAM Phase 2 — Score with LLM Judge

Uses the local Gemma model as judge (free, no rate limits). Takes ~1-3 hours for 4 samples.

If interrupted, re-run with `--resume` to skip already-scored questions.

In [ ]:
import time

BEAM_PY = '/opt/conda/envs/beam/bin/python'

start = time.time()
print(f'Starting BEAM Phase 2 (Scoring) at {time.strftime("%H:%M:%S")}')
print(f'Judge model: google/gemma-4-31B-it (local gpu02)')
print('=' * 60)

!{BEAM_PY} evaluation/BEAM/score.py \
    --responses {RESPONSES_PATH} \
    --output {SCORES_PATH} \
    --judge-model google/gemma-4-31B-it \
    --resume

elapsed = time.time() - start
print(f'\n{"=" * 60}')
print(f'Phase 2 complete in {elapsed/3600:.1f} hours')
print(f'Scores saved to: {SCORES_PATH}')

## 8. View aggregate results

In [ ]:
import json
from pathlib import Path

scores_path = Path(SCORES_PATH)
if scores_path.exists():
    with open(scores_path) as f:
        data = json.load(f)
    
    print('=' * 60)
    print('BEAM Aggregate Scores')
    print('=' * 60)
    print(json.dumps(data.get('aggregate', data), indent=2))
else:
    print(f'Scores file not found at {scores_path}')
    print('Check if Phase 2 completed successfully.')

## 9. Resume interrupted run

If JupyterHub disconnected mid-run, use this cell to resume.

- **Checkpoints** (`--checkpoint-dir + --resume`): Skips completed pipeline stages (entity extraction, relation extraction, evidence linking, etc.) within each conversation.
- **KG cache** (`--save-kg-dir`): Completed conversations' KGs are saved. Use `--load-kg-dir` to skip KG building entirely for those.

Both mechanisms work together: checkpoints resume mid-conversation, KG cache skips fully-built conversations.

In [ ]:
BEAM_PY = '/opt/conda/envs/beam/bin/python'

# Resume Phase 1 — continues from last completed checkpoint stage.
# If a conversation's KG was fully built and saved, it will be loaded
# from --save-kg-dir instead of rebuilt.
!{BEAM_PY} evaluation/BEAM/run_eval.py \
    --tier {TIER} \
    --max-samples {MAX_SAMPLES} \
    --max-questions {MAX_QUESTIONS} \
    --chunk-size {CHUNK_SIZE} \
    --retrieval {RETRIEVAL} \
    --model google/gemma-4-31B-it \
    --embedding-model accounts/fireworks/models/qwen3-embedding-8b \
    --save-kg-dir {KG_DIR} \
    --load-kg-dir {KG_DIR} \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --resume \
    --output {RESPONSES_PATH}

print('\n=== Resume complete ===')

## 10. Check GPU status (optional)

Monitor GPU utilization on gpu02 while the benchmark runs.

In [ ]:
# Check GPU status and vLLM log from a JupyterHub terminal:
#
#   ssh gpu02.mind.cs.umd.edu 'nvidia-smi'
#   ssh gpu02.mind.cs.umd.edu 'tail -f /scratch/vllm_gemma.log'
#
# Or just poll vLLM's health endpoint from here:
import requests

try:
    resp = requests.get(f'{VLLM_URL}/v1/models', timeout=5)
    if resp.status_code == 200:
        models = resp.json().get('data', [])
        print(f'✅ vLLM alive — {len(models)} model(s): {[m["id"] for m in models]}')
    else:
        print(f'⚠️  vLLM returned {resp.status_code}')
except Exception as e:
    print(f'❌ vLLM unreachable: {e}')